## Data prep


In [ ]:
!pip -q install -U "transformers>=4.41.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
                 "trl>=0.11.0" "peft>=0.11.1" "bitsandbytes>=0.46.1" "safetensors>=0.4.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.3 MB/s eta 0:00:00


In [ ]:
COUNTRY_CODES = ["US"]
# Pass a list of country abbreviations here.
# This must match the abbreviation used in the paths.

# Add as many countries as you like, but keep the computational constraints in mind
# The preprocessing takes a while, could be around 40 mins per country
# Start with one and see how it goes

def resolve_country_codes(country_code):
    if country_code == "US":
        return "USA", "US"
    return country_code, country_code


In [ ]:
from pathlib import Path

# path to raw files
DATA_DIR = Path("/content/drive/MyDrive/DPO")
# path to train/eval files
DATA_DIR1 = Path("/content/drive/MyDrive/DPO/pre-processing")

def build_country_paths(country_code):
    data_country_code, adapter_tag = resolve_country_codes(country_code)

    raw_file = DATA_DIR1 / f"{adapter_tag}_triplets_3594_3.jsonl"
    train_file = DATA_DIR / f"{data_country_code}_3594_train_3.jsonl"
    eval_file = DATA_DIR / f"{data_country_code}_3594_eval_3.jsonl"

    # for pre-processing data
    out_file = DATA_DIR / f"{data_country_code}_3594_train_with_ref_3.jsonl"

    return raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag


In [ ]:
import os

# Must be set BEFORE importing torch/transformers/trl/accelerate
os.environ["ACCELERATE_MIXED_PRECISION"] = "fp16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

os.environ["TRANSFORMERS_NO_BF16"] = "1"   # hard-disable bf16 paths in transformers


import torch
torch.set_default_dtype(torch.float16)

print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

CUDA: 12.8
GPU: Tesla T4


In [ ]:
#!pip install trl
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model#, prepare_model_for_kbit_training
from trl import DPOTrainer, DPOConfig

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

In [ ]:
# login with hugging face.
# must have an account with them
# occasionally also requests an access token
#!pip install -U "huggingface_hub[cli]"
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import random
from pathlib import Path


#train-test split here. adjust as needed
TRAIN_FRAC = 0.80
SEED = 42


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def split_country_file(raw_path, train_path, eval_path, country, train_frac=0.80, seed=42):
    rows = load_jsonl(raw_path)

    # Add useful metadata for later validation.
    for i, row in enumerate(rows):
        row.setdefault("country", country)
        row.setdefault("item_id", f"{country}_{i:04d}")

    rng = random.Random(seed)
    rng.shuffle(rows)

    n_train = int(len(rows) * train_frac)

    train_rows = rows[:n_train]
    eval_rows = rows[n_train:]

    write_jsonl(train_rows, train_path)
    write_jsonl(eval_rows, eval_path)

    print(f"{country}: {len(rows)} total")
    print(f"  train: {len(train_rows)} -> {train_path}")
    print(f"  eval:  {len(eval_rows)} -> {eval_path}")

    return train_rows, eval_rows


for country_code in COUNTRY_CODES:
    raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag = build_country_paths(country_code)

    train, eval = split_country_file(
        raw_path=raw_file,
        train_path=train_file,
        eval_path=eval_file,
        country=data_country_code,
        train_frac=TRAIN_FRAC,
        seed=SEED,
    )


'\nus_train, us_eval = split_country_file(\n    raw_path=US_RAW_FILE,\n    train_path=US_TRAIN_FILE,\n    eval_path=US_EVAL_FILE,\n    country="US",\n    train_frac=TRAIN_FRAC,\n    seed=SEED,\n)\n\nmex_train, mex_eval = split_country_file(\n    raw_path=MEX_RAW_FILE,\n    train_path=MEX_TRAIN_FILE,\n    eval_path=MEX_EVAL_FILE,\n    country="Mexico",\n    train_frac=TRAIN_FRAC,\n    seed=SEED,\n)\n'

In [ ]:

import json, math, os

# Keep these modest for Colab
MAX_PROMPT_TOKENS = 256
MAX_COMPLETION_TOKENS = 256   # chosen/rejected truncation
BATCH_SIZE = 1                # increase only if you have VRAM headroom

device = "cuda" if torch.cuda.is_available() else "cpu"

compute_dtype = torch.float16  # avoid bf16 issues on Colab

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading reference (base) model in 4-bit...")
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},              # force to GPU for this pass
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
)
ref_model.eval()

def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )


def format_dataset_example(ex):
    ex["prompt"] = format_prompt_text(tokenizer, ex["prompt"])
    return ex

@torch.no_grad()
def seq_logprob_for_completion(prompt_text: str, completion_text: str) -> float:

    #Returns log p(completion | prompt) under ref_model, summed over completion tokens.
    #Prompt tokens are masked out.

    prompt = format_prompt_text(tokenizer, prompt_text)

    # Tokenize prompt with truncation (keep end tends to preserve immediate context)
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    if len(prompt_ids) > MAX_PROMPT_TOKENS:
        prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

    # Tokenize completion
    comp_ids = tokenizer(completion_text, add_special_tokens=False).input_ids
    if len(comp_ids) > MAX_COMPLETION_TOKENS:
        comp_ids = comp_ids[:MAX_COMPLETION_TOKENS]

    input_ids = prompt_ids + comp_ids
    if len(input_ids) < 2:
        return float("-inf")

    # Build labels: ignore prompt tokens, score completion tokens
    labels = ([-100] * len(prompt_ids)) + comp_ids

    # Convert to tensors
    input_ids_t = torch.tensor([input_ids], device=ref_model.device)
    labels_t    = torch.tensor([labels], device=ref_model.device)

    outputs = ref_model(input_ids=input_ids_t)
    logits = outputs.logits  # [1, seq, vocab]

    log_probs = torch.log_softmax(logits, dim=-1)

    # Shift for next-token prediction
    log_probs = log_probs[:, :-1, :]
    labels_shifted = labels_t[:, 1:]

    mask = labels_shifted.ne(-100)
    # gather logp for gold tokens
    gold = labels_shifted.clone()
    gold[~mask] = 0
    token_logps = log_probs.gather(-1, gold.unsqueeze(-1)).squeeze(-1)
    token_logps = token_logps * mask

    return float(token_logps.sum().detach().cpu().to(torch.float32))


'\n# Keep these modest for Colab\nMAX_PROMPT_TOKENS = 256\nMAX_COMPLETION_TOKENS = 256   # chosen/rejected truncation\nBATCH_SIZE = 1                # increase only if you have VRAM headroom\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\n\ncompute_dtype = torch.float16  # avoid bf16 issues on Colab\n\nbnb_config = BitsAndBytesConfig(\n    load_in_4bit=True,\n    bnb_4bit_quant_type="nf4",\n    bnb_4bit_compute_dtype=compute_dtype,\n    bnb_4bit_use_double_quant=True,\n)\n\nprint("Loading tokenizer...")\ntokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\n\nprint("Loading reference (base) model in 4-bit...")\nref_model = AutoModelForCausalLM.from_pretrained(\n    MODEL_NAME,\n    quantization_config=bnb_config,\n    device_map={"": 0},              # force to GPU for this pass\n    torch_dtype=compute_dtype,\n    low_cpu_mem_usage=True,\n)\nref_model.eval()\n\ndef build_user_pr

In [ ]:
def build_user_prompt(prompt_text: str) -> str:
    return (
        "You are answering a questionnaire as an individual person. "
        "Respond naturally and thoughtfully, as someone would in real life. "
        "Do not mention being an AI or assistant. "
        "Keep the answer short, under 3 sentences. "
        "Give a sincere, human-like answer.\n\n"
        "Situation:\n"
        f"{prompt_text.strip()}\n\n"
        "Answer:"
    )


def format_prompt_text(tokenizer, prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": build_user_prompt(prompt_text)}],
        tokenize=False,
        add_generation_prompt=True,
    )


def format_dataset_example(ex):
    ex["prompt"] = format_prompt_text(tokenizer, ex["prompt"])
    return ex

@torch.no_grad()
def seq_logprob_for_completion(prompt_text: str, completion_text: str) -> float:
    """
    Returns log p(completion | prompt) under ref_model, summed over completion tokens.
    Prompt tokens are masked out.
    """
    prompt = format_prompt_text(tokenizer, prompt_text)

    # Tokenize prompt with truncation (keep end tends to preserve immediate context)
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids
    if len(prompt_ids) > MAX_PROMPT_TOKENS:
        prompt_ids = prompt_ids[-MAX_PROMPT_TOKENS:]

    # Tokenize completion
    comp_ids = tokenizer(completion_text, add_special_tokens=False).input_ids
    if len(comp_ids) > MAX_COMPLETION_TOKENS:
        comp_ids = comp_ids[:MAX_COMPLETION_TOKENS]

    input_ids = prompt_ids + comp_ids
    if len(input_ids) < 2:
        return float("-inf")

    # Build labels: ignore prompt tokens, score completion tokens
    labels = ([-100] * len(prompt_ids)) + comp_ids

    # Convert to tensors
    input_ids_t = torch.tensor([input_ids], device=ref_model.device)
    labels_t    = torch.tensor([labels], device=ref_model.device)

    outputs = ref_model(input_ids=input_ids_t)
    logits = outputs.logits  # [1, seq, vocab]

    log_probs = torch.log_softmax(logits, dim=-1)

    # Shift for next-token prediction
    log_probs = log_probs[:, :-1, :]
    labels_shifted = labels_t[:, 1:]

    mask = labels_shifted.ne(-100)
    # gather logp for gold tokens
    gold = labels_shifted.clone()
    gold[~mask] = 0
    token_logps = log_probs.gather(-1, gold.unsqueeze(-1)).squeeze(-1)
    token_logps = token_logps * mask

    return float(token_logps.sum().detach().cpu().to(torch.float32))



In [ ]:
# Read all data
all_data = {}
for country_code in COUNTRY_CODES:
    raw_file, train_file, eval_file, out_file, data_country_code, adapter_tag = build_country_paths(country_code)
    with open(train_file, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]

    all_data[country_code] = (data, out_file)
    print(f"Loaded {len(data)} examples from {train_file}")


'\nwith open(IN_FILE, "r", encoding="utf-8") as f:\n    data = [json.loads(line) for line in f if line.strip()]\n\nprint(f"Loaded {len(data)} examples from {IN_FILE}")\n'

In [ ]:
# Compute and write
# takes a while

for country_code in COUNTRY_CODES:
    data, out_file = all_data[country_code]

    with open(out_file, "w", encoding="utf-8") as out:
        for i, ex in enumerate(data, 1):
            p = ex["prompt"]
            chosen = ex["chosen"]
            rejected = ex["rejected"]

            ref_chosen_logps = seq_logprob_for_completion(p, chosen)
            ref_rejected_logps = seq_logprob_for_completion(p, rejected)

            ex["ref_chosen_logps"] = ref_chosen_logps
            ex["ref_rejected_logps"] = ref_rejected_logps

            out.write(json.dumps(ex, ensure_ascii=False) + "\n")

            if i % 25 == 0:
                print(f"Precomputed {i}/{len(data)}")

    print("Done. Wrote:", out_file)

# Free VRAM
del ref_model
torch.cuda.empty_cache()


Precomputed 25/2875
Precomputed 50/2875
Precomputed 75/2875
Precomputed 100/2875
Precomputed 125/2875
Precomputed 150/2875
Precomputed 175/2875
Precomputed 200/2875
Precomputed 225/2875
Precomputed 250/2875
Precomputed 275/2875
Precomputed 300/2875
Precomputed 325/2875
Precomputed 350/2875
Precomputed 375/2875
Precomputed 400/2875
Precomputed 425/2875
Precomputed 450/2875
Precomputed 475/2875
Precomputed 500/2875
Precomputed 525/2875
Precomputed 550/2875
Precomputed 575/2875
Precomputed 600/2875
Precomputed 625/2875
Precomputed 650/2875
Precomputed 675/2875
Precomputed 700/2875
Precomputed 725/2875
Precomputed 750/2875
Precomputed 775/2875
Precomputed 800/2875
Precomputed 825/2875
Precomputed 850/2875
Precomputed 875/2875
Precomputed 900/2875
Precomputed 925/2875
Precomputed 950/2875
Precomputed 975/2875
Precomputed 1000/2875
Precomputed 1025/2875
Precomputed 1050/2875
Precomputed 1075/2875
Precomputed 1100/2875
Precomputed 1125/2875
Precomputed 1150/2875
Precomputed 1175/2875
Precompu